# Data Deduplication Lab - Getting Started

Welcome to the Data Deduplication Lab. This notebook prepares your Cloudera AI Workbench session for the Phase 1 exercises.

## Learning Objectives

By the end of this notebook, you will:
- Create a Spark session in Cloudera AI Workbench
- Generate a sample customer dataset with intentional duplicates
- Save that dataset to HDFS under `/tmp` for later exercises

## What this notebook does

1. **Spark setup** — create and verify a Spark session
2. **Sample data** — build a small CSV-style dataset with duplicate records
3. **HDFS write** — save the sample file to `hdfs:///tmp/` for Exercise 1+

## Prerequisites

- Access to Cloudera AI Workbench with Spark enabled
- Permission to write to HDFS `/tmp`
- Basic Python familiarity

## Types of Deduplication (lab overview)

### Record-Level Deduplication (Exercises 1-3)
- Removes duplicate **rows/records** within a dataset
- Example: two customer rows with the same name and email
- Covered in Exercises 1, 2, and 3

### File-Level Deduplication (Exercise 4)
- Finds duplicate **files** based on content hashes
- Covered in Exercise 4

**Next notebook after this setup:** `01_Basic_Deduplication.ipynb`


## 1. Create Spark Session

In Cloudera AI Workbench, PySpark is typically pre-installed and configured for the cluster. The cell below creates a Spark session suitable for this lab.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Create Spark session for Cloudera AI Workbench
spark = (
    SparkSession.builder
    .appName("DeduplicationLab_GettingStarted")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Default FS:    {spark.sparkContext._jsc.hadoopConfiguration().get('fs.defaultFS')}")
print("✓ Spark session created successfully")


## 2. Create Sample Dataset

Generate a customer-style dataset with intentional duplicates on `name` + `email`. This is the input used by later notebooks.


In [ ]:
# Sample records: unique customers + intentional duplicates
base_records = [
    ("Alice Johnson", "alice.johnson@example.com", "New York", 34),
    ("Bob Smith", "bob.smith@example.com", "Chicago", 41),
    ("Carol Lee", "carol.lee@example.com", "Austin", 29),
    ("David Kim", "david.kim@example.com", "Seattle", 37),
    ("Eva Martinez", "eva.martinez@example.com", "Miami", 45),
    ("Frank Wright", "frank.wright@example.com", "Denver", 52),
    ("Grace Chen", "grace.chen@example.com", "Boston", 31),
    ("Henry Patel", "henry.patel@example.com", "Atlanta", 28),
    ("Ivy Nguyen", "ivy.nguyen@example.com", "Portland", 39),
    ("Jack Brown", "jack.brown@example.com", "Dallas", 44),
]

# Duplicate a subset of records (same name+email, possibly different city/age)
duplicates = [
    ("Alice Johnson", "alice.johnson@example.com", "New York", 34),  # exact duplicate
    ("Bob Smith", "bob.smith@example.com", "Chicago", 41),            # exact duplicate
    ("Carol Lee", "carol.lee@example.com", "Austin", 30),             # same keys, different age
    ("David Kim", "david.kim@example.com", "Seattle", 37),            # exact duplicate
    ("Eva Martinez", "eva.martinez@example.com", "Orlando", 45),      # same keys, different city
]

# Expand to ~1000 rows by cycling base records, then append duplicates
records = []
target_rows = 1000
while len(records) < target_rows - len(duplicates):
    records.extend(base_records)
records = records[: target_rows - len(duplicates)] + duplicates

schema = StructType([
    StructField("name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("city", StringType(), True),
    StructField("age", IntegerType(), True),
])

df = spark.createDataFrame(records, schema=schema)

total_count = df.count()
unique_count = df.select("name", "email").distinct().count()
duplicates_count = total_count - unique_count

print(f"Total records: {total_count:,}")
print(f"Unique (name+email): {unique_count:,}")
print(f"Duplicate records: {duplicates_count:,}")
print(f"Duplicate rate: {(duplicates_count / total_count * 100):.2f}%")
print("\nPreview:")
df.show(10, truncate=False)
df.printSchema()


## 3. Save Sample File to HDFS `/tmp`

Write the sample dataset to HDFS so later notebooks can read a shared cluster path.

**Output path:** `hdfs:///tmp/cdp_user_demo/phase1/redundant_data.csv`

> Note: Spark writes CSV as a **directory** containing `part-*.csv` files. That is expected and is what Exercise notebooks should read.


In [ ]:
# HDFS destination under /tmp
HDFS_OUTPUT_DIR = "hdfs:///tmp/cdp_user_demo/phase1/redundant_data.csv"

(
    df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(HDFS_OUTPUT_DIR)
)

print(f"✓ Sample data written to: {HDFS_OUTPUT_DIR}")

# Verify by reading back from HDFS
df_hdfs = spark.read.csv(HDFS_OUTPUT_DIR, header=True, inferSchema=True)
print(f"✓ Verified read from HDFS — rows: {df_hdfs.count():,}")
print(f"  Columns: {', '.join(df_hdfs.columns)}")
df_hdfs.show(5, truncate=False)


## 4. Quick Duplicate Check

Confirm the HDFS file still contains the expected duplicate pattern before moving on.


In [ ]:
total_count = df_hdfs.count()
unique_count = df_hdfs.select("name", "email").distinct().count()
duplicates_count = total_count - unique_count
duplicate_rate = (duplicates_count / total_count * 100) if total_count else 0

print(f"Total records: {total_count:,}")
print(f"Unique records (by name+email): {unique_count:,}")
print(f"Duplicate records: {duplicates_count:,}")
print(f"Duplicate rate: {duplicate_rate:.2f}%")
print(f"\nShared path for later notebooks:\n  {HDFS_OUTPUT_DIR}")


## Next Steps

Setup is complete. Use this HDFS path in subsequent notebooks:

```text
hdfs:///tmp/cdp_user_demo/phase1/redundant_data.csv
```

1. **Exercise 1**: Basic Deduplication — `01_Basic_Deduplication.ipynb`
2. **Exercise 2**: Iceberg REST Catalog — `02_Iceberg_REST_Catalog.ipynb`
3. **Exercise 3**: Approximate Methods — `03_Approximate_Methods.ipynb`
4. **Exercise 4**: File-Level Deduplication — `04_File_Level_Deduplication.ipynb`

## Cleanup

Stop the Spark session when you are finished with this notebook. Later notebooks create their own sessions.


In [ ]:
spark.stop()
print("✓ Spark session stopped")
